### Source Tables:
- _exponent._bronze_allscripts_tw_works.dbo_vendor_item
- _exponent._bronze_allscripts_tw_works.dbo_vendor_item_extension (optional)

### Strategy:
- Extract conditions/diagnoses from dbo_vendor_item filtered by ItemType
- Map SnomedCode to OMOP concept_id via vocabulary tables
- Map StatusQualifier to condition_status_concept_id via domain_source_to_concept
- Calculate condition_end_date from Duration + DurationUnit when available
- Use condition_type_concept_id = 32817 (EHR encounter record)

### Notes:
- This notebook depends on source_to_person being populated
- provider_id and visit_occurrence_id will be NULL (mapping tables not yet created)
- SNOMED concept mapping requires OMOP vocabulary tables to be loaded
- condition_source_concept_id uses 0 if SNOMED code not found in vocabulary

In [0]:
# %sql
# -- Insert matched SNOMED condition mappings into domain_source_to_concept
# INSERT INTO _exponent.omop_mapping.domain_source_to_concept (
#     source_system,
#     source_table,
#     source_field,
#     domain_id,
#     source_id,
#     source_value,
#     omop_concept_id,
#     active_flag,
#     last_update_tsp
# )
# SELECT DISTINCT
#     'allscripts_tw' AS source_system,
#     'dbo_vendor_item' AS source_table,
#     'SnomedCode' AS source_field,
#     'Condition' AS domain_id,
#     vi.SnomedCode AS source_id,
#     c.concept_name AS source_value,
#     c.concept_id AS omop_concept_id,
#     1 AS active_flag,
#     CURRENT_TIMESTAMP() AS last_update_tsp
# FROM _exponent._bronze_allscripts_tw_works.dbo_vendor_item vi
# INNER JOIN _exponent.omop.concept c
#     ON c.concept_code = vi.SnomedCode
#     AND c.vocabulary_id = 'SNOMED'
#     AND c.standard_concept = 'S'
#     AND c.domain_id = 'Condition'
# WHERE vi.SnomedCode IS NOT NULL

In [0]:
source = 'allscripts_tw'

# Transformation

In [0]:
silver_condition_occurrence_df = spark.sql(f'''
SELECT
  -- Concept mapping: SNOMED code to standard OMOP concept
  -- First try to find standard concept, fallback to 0 if not mapped
  COALESCE(snomed_concept.concept_id, 0) AS condition_concept_id,
  
  -- Dates
  DATE(COALESCE(vi.PerformedDTTM, vi.RecordedDTTM, vi.CreateDTTM)) AS condition_start_date,
  COALESCE(vi.PerformedDTTM, vi.RecordedDTTM, vi.CreateDTTM) AS condition_start_datetime,
  
  -- End date calculation from Duration + DurationUnit
  CASE 
    WHEN vi.Duration IS NOT NULL AND vi.DurationUnit IS NOT NULL THEN
      CASE vi.DurationUnit
        WHEN 'D' THEN DATE_ADD(DATE(COALESCE(vi.PerformedDTTM, vi.RecordedDTTM, vi.CreateDTTM)), CAST(vi.Duration AS INT))
        WHEN 'W' THEN DATE_ADD(DATE(COALESCE(vi.PerformedDTTM, vi.RecordedDTTM, vi.CreateDTTM)), CAST(vi.Duration AS INT) * 7)
        WHEN 'M' THEN ADD_MONTHS(DATE(COALESCE(vi.PerformedDTTM, vi.RecordedDTTM, vi.CreateDTTM)), CAST(vi.Duration AS INT))
        WHEN 'Y' THEN ADD_MONTHS(DATE(COALESCE(vi.PerformedDTTM, vi.RecordedDTTM, vi.CreateDTTM)), CAST(vi.Duration AS INT) * 12)
        ELSE NULL
      END
    ELSE NULL
  END AS condition_end_date,
  
  CASE 
    WHEN vi.Duration IS NOT NULL AND vi.DurationUnit IS NOT NULL THEN
      CASE vi.DurationUnit
        WHEN 'D' THEN CAST(DATE_ADD(DATE(COALESCE(vi.PerformedDTTM, vi.RecordedDTTM, vi.CreateDTTM)), CAST(vi.Duration AS INT)) AS TIMESTAMP)
        WHEN 'W' THEN CAST(DATE_ADD(DATE(COALESCE(vi.PerformedDTTM, vi.RecordedDTTM, vi.CreateDTTM)), CAST(vi.Duration AS INT) * 7) AS TIMESTAMP)
        WHEN 'M' THEN CAST(ADD_MONTHS(DATE(COALESCE(vi.PerformedDTTM, vi.RecordedDTTM, vi.CreateDTTM)), CAST(vi.Duration AS INT)) AS TIMESTAMP)
        WHEN 'Y' THEN CAST(ADD_MONTHS(DATE(COALESCE(vi.PerformedDTTM, vi.RecordedDTTM, vi.CreateDTTM)), CAST(vi.Duration AS INT) * 12) AS TIMESTAMP)
        ELSE NULL
      END
    ELSE NULL
  END AS condition_end_datetime,
  
  -- Type concept: EHR encounter record
  32817 AS condition_type_concept_id,
  
  -- Status concept mapping via domain_source_to_concept
  COALESCE(status_concept.omop_concept_id, 0) AS condition_status_concept_id,
  
  -- Stop reason (if status indicates resolved/inactive)
  CASE 
    WHEN UPPER(vi.StatusQualifier) IN ('RESOLVED', 'INACTIVE', 'COMPLETED') THEN vi.StatusQualifier
    ELSE NULL 
  END AS stop_reason,
  
  -- Source values
  COALESCE(vi.SnomedCode, vi.LOINCCode, vi.VendorID, vi.NomenclatureID) AS condition_source_value,
  COALESCE(snomed_source_concept.concept_id, 0) AS condition_source_concept_id,
  vi.StatusQualifier AS condition_status_source_value,
  
  -- FK source values (resolved to IDs in gold layer)
  CONCAT('{source}', ' | ', CAST(vi.PatientID AS STRING)) AS person_source_value,
  CASE 
    WHEN vi.WhoDidItID IS NOT NULL THEN CONCAT('{source}', ' | ', CAST(vi.WhoDidItID AS STRING))
    ELSE NULL 
  END AS provider_source_value,
  CASE 
    WHEN vi.EncounterID IS NOT NULL THEN CONCAT('{source}', ' | ', CAST(vi.EncounterID AS STRING))
    ELSE NULL 
  END AS visit_occurrence_source_value,
  NULL AS visit_detail_source_value,
  
  -- Unique identifier for this condition occurrence
  CONCAT('{source}', ' | ', CAST(vi.ID AS STRING)) AS condition_occurrence_source_value,
  '{source}' AS source_system

FROM _exponent._bronze_allscripts_tw_works.dbo_vendor_item vi

-- Join to OMOP concept table for SNOMED standard concept
LEFT OUTER JOIN _exponent.omop.concept snomed_concept
  ON snomed_concept.concept_code = vi.SnomedCode
  AND snomed_concept.vocabulary_id = 'SNOMED'
  AND snomed_concept.standard_concept = 'S'
  AND snomed_concept.domain_id = 'Condition'

-- Join to OMOP concept table for source concept (non-standard)
LEFT OUTER JOIN _exponent.omop.concept snomed_source_concept
  ON snomed_source_concept.concept_code = vi.SnomedCode
  AND snomed_source_concept.vocabulary_id = 'SNOMED'

-- Join to domain_source_to_concept for status mapping
LEFT OUTER JOIN _exponent.omop_mapping.domain_source_to_concept status_concept
  ON UPPER(status_concept.source_value) = UPPER(vi.StatusQualifier)
  AND status_concept.domain_id = 'ConditionStatus'
  AND status_concept.source_system = '{source}'

-- Only include patients that exist in source_to_person mapping
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT('{source}', ' | ', CAST(vi.PatientID AS STRING))
  AND stp.active_flag = TRUE

WHERE vi.IsErrorFLAG = 'N'
  AND vi.PatientID IS NOT NULL
  AND COALESCE(vi.PerformedDTTM, vi.RecordedDTTM, vi.CreateDTTM) IS NOT NULL
  -- Filter to condition-related item types
  -- Note: Adjust these based on actual ItemType values in your data
  AND (
    vi.SnomedCode IS NOT NULL  -- Has SNOMED code (likely a condition)
    OR UPPER(vi.ItemType) IN ('DX', 'DIAGNOSIS', 'PROBLEM', 'CONDITION', 'HX', 'PMH')
  )
''')

display(silver_condition_occurrence_df)
silver_condition_occurrence_df.createOrReplaceTempView("silver_condition_occurrence")

In [0]:
%sql
-- Check distinct ItemType values to understand what data exists
SELECT DISTINCT ItemType, COUNT(*) as cnt
FROM _exponent._bronze_allscripts_tw_works.dbo_vendor_item
WHERE IsErrorFLAG = 'N'
GROUP BY ItemType
ORDER BY cnt DESC

In [0]:
%sql
-- Merge to Silver layer
MERGE INTO _exponent.omop_silver.condition_occurrence AS t
USING silver_condition_occurrence AS s
ON t.condition_occurrence_source_value = s.condition_occurrence_source_value

WHEN MATCHED AND (
     NOT (t.condition_concept_id <=> s.condition_concept_id)
  OR NOT (t.condition_start_date <=> s.condition_start_date)
  OR NOT (t.condition_start_datetime <=> s.condition_start_datetime)
  OR NOT (t.condition_end_date <=> s.condition_end_date)
  OR NOT (t.condition_end_datetime <=> s.condition_end_datetime)
  OR NOT (t.condition_type_concept_id <=> s.condition_type_concept_id)
  OR NOT (t.condition_status_concept_id <=> s.condition_status_concept_id)
  OR NOT (t.stop_reason <=> s.stop_reason)
  OR NOT (t.condition_source_value <=> s.condition_source_value)
  OR NOT (t.condition_source_concept_id <=> s.condition_source_concept_id)
  OR NOT (t.condition_status_source_value <=> s.condition_status_source_value)
  OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.provider_source_value <=> s.provider_source_value)
  OR NOT (t.visit_occurrence_source_value <=> s.visit_occurrence_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.condition_concept_id          = s.condition_concept_id,
  t.condition_start_date          = s.condition_start_date,
  t.condition_start_datetime      = s.condition_start_datetime,
  t.condition_end_date            = s.condition_end_date,
  t.condition_end_datetime        = s.condition_end_datetime,
  t.condition_type_concept_id     = s.condition_type_concept_id,
  t.condition_status_concept_id   = s.condition_status_concept_id,
  t.stop_reason                   = s.stop_reason,
  t.condition_source_value        = s.condition_source_value,
  t.condition_source_concept_id   = s.condition_source_concept_id,
  t.condition_status_source_value = s.condition_status_source_value,
  t.person_source_value           = s.person_source_value,
  t.provider_source_value         = s.provider_source_value,
  t.visit_occurrence_source_value = s.visit_occurrence_source_value,
  t.visit_detail_source_value     = s.visit_detail_source_value,
  t.source_system                 = s.source_system,
  t.last_mod_tsp                  = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value,
  person_source_value,
  provider_source_value,
  visit_occurrence_source_value,
  visit_detail_source_value,
  condition_occurrence_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.condition_concept_id,
  s.condition_start_date,
  s.condition_start_datetime,
  s.condition_end_date,
  s.condition_end_datetime,
  s.condition_type_concept_id,
  s.condition_status_concept_id,
  s.stop_reason,
  s.condition_source_value,
  s.condition_source_concept_id,
  s.condition_status_source_value,
  s.person_source_value,
  s.provider_source_value,
  s.visit_occurrence_source_value,
  s.visit_detail_source_value,
  s.condition_occurrence_source_value,
  s.source_system,
  current_timestamp()
);

In [0]:
%sql
-- Verify silver layer
SELECT * FROM _exponent.omop_silver.condition_occurrence LIMIT 10

In [0]:
%sql
-- Insert new mappings to source_to_condition_occurrence
INSERT INTO _exponent.omop_mapping.source_to_condition_occurrence (
    source_system,
    condition_occurrence_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.condition_occurrence_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, condition_occurrence_source_value, last_mod_tsp
    FROM _exponent.omop_silver.condition_occurrence
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_condition_occurrence x
  ON s.condition_occurrence_source_value = x.condition_occurrence_source_value;

In [0]:
%sql
-- Verify mapping table
SELECT * FROM _exponent.omop_mapping.source_to_condition_occurrence LIMIT 10

In [0]:
%sql
-- Merge to Gold layer
-- Note: provider_id and visit_occurrence_id are NULL (mapping tables not created)
MERGE INTO _exponent.omop.condition_occurrence AS gold
USING (
  SELECT
    sco.condition_occurrence_id,
    stp.person_id,
    s.condition_concept_id,
    s.condition_start_date,
    s.condition_start_datetime,
    s.condition_end_date,
    s.condition_end_datetime,
    s.condition_type_concept_id,
    s.condition_status_concept_id,
    s.stop_reason,
    NULL AS provider_id,  -- source_to_provider not yet created
    NULL AS visit_occurrence_id,  -- source_to_visit_occurrence not yet created
    NULL AS visit_detail_id,  -- source_to_visit_detail not yet created
    s.condition_source_value,
    s.condition_source_concept_id,
    s.condition_status_source_value
  FROM _exponent.omop_silver.condition_occurrence s
  JOIN _exponent.omop_mapping.source_to_condition_occurrence sco
    ON sco.condition_occurrence_source_value = s.condition_occurrence_source_value
   AND sco.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.active_flag = TRUE
) AS src
ON gold.condition_occurrence_id = src.condition_occurrence_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                     = src.person_id,
  gold.condition_concept_id          = src.condition_concept_id,
  gold.condition_start_date          = src.condition_start_date,
  gold.condition_start_datetime      = src.condition_start_datetime,
  gold.condition_end_date            = src.condition_end_date,
  gold.condition_end_datetime        = src.condition_end_datetime,
  gold.condition_type_concept_id     = src.condition_type_concept_id,
  gold.condition_status_concept_id   = src.condition_status_concept_id,
  gold.stop_reason                   = src.stop_reason,
  gold.provider_id                   = src.provider_id,
  gold.visit_occurrence_id           = src.visit_occurrence_id,
  gold.visit_detail_id               = src.visit_detail_id,
  gold.condition_source_value        = src.condition_source_value,
  gold.condition_source_concept_id   = src.condition_source_concept_id,
  gold.condition_status_source_value = src.condition_status_source_value

WHEN NOT MATCHED THEN INSERT (
  condition_occurrence_id,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value
)
VALUES (
  src.condition_occurrence_id,
  src.person_id,
  src.condition_concept_id,
  src.condition_start_date,
  src.condition_start_datetime,
  src.condition_end_date,
  src.condition_end_datetime,
  src.condition_type_concept_id,
  src.condition_status_concept_id,
  src.stop_reason,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.condition_source_value,
  src.condition_source_concept_id,
  src.condition_status_source_value
);

In [0]:
%sql
-- Verify gold layer
SELECT * FROM _exponent.omop.condition_occurrence LIMIT 10

In [0]:
%sql
-- Validation: Count records at each layer
SELECT 'Silver' AS layer, COUNT(*) AS record_count FROM _exponent.omop_silver.condition_occurrence
UNION ALL
SELECT 'Mapping' AS layer, COUNT(*) AS record_count FROM _exponent.omop_mapping.source_to_condition_occurrence
UNION ALL
SELECT 'Gold' AS layer, COUNT(*) AS record_count FROM _exponent.omop.condition_occurrence

In [0]:
%sql
-- Data Quality: Check unmapped conditions (concept_id = 0)
SELECT 
  COUNT(*) AS total_records,
  SUM(CASE WHEN condition_concept_id = 0 THEN 1 ELSE 0 END) AS unmapped_count,
  ROUND(100.0 * SUM(CASE WHEN condition_concept_id = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS unmapped_pct
FROM _exponent.omop.condition_occurrence